# **Mount Google Drive**

In [45]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Set project paths and verify saved files**

In [46]:
from pathlib import Path
import os
import sys

PROJECT_DIR = Path("/content/drive/MyDrive/Camouflaged-Wildlife-Detection-System")
APP_PATH = PROJECT_DIR / "app.py"
MODEL_FILE = PROJECT_DIR / "models" / "unet.py"
CHECKPOINT_PATH = PROJECT_DIR / "models" / "best_model.pth"

required_paths = {
    "Project folder": PROJECT_DIR,
    "Gradio app": APP_PATH,
    "Model architecture": MODEL_FILE,
    "Best checkpoint": CHECKPOINT_PATH,
}

for label, path in required_paths.items():
    print(f"{label}: {path.exists()} — {path}")

missing = [label for label, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required project items: " + ", ".join(missing))

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("\nCheckpoint size:", round(CHECKPOINT_PATH.stat().st_size / (1024**2), 2), "MB")

Project folder: True — /content/drive/MyDrive/Camouflaged-Wildlife-Detection-System
Gradio app: True — /content/drive/MyDrive/Camouflaged-Wildlife-Detection-System/app.py
Model architecture: True — /content/drive/MyDrive/Camouflaged-Wildlife-Detection-System/models/unet.py
Best checkpoint: True — /content/drive/MyDrive/Camouflaged-Wildlife-Detection-System/models/best_model.pth

Checkpoint size: 280.8 MB


# **Install Gradio**

In [47]:
%pip -q install gradio==6.31.0

import gradio as gr
import torch

print("Gradio:", gr.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Gradio: 6.31.0
PyTorch: 2.14.1+cu130
CUDA available: True


# **Checkpoint metadata**

In [48]:
import ast

app_source = APP_PATH.read_text()
ast.parse(app_source)
print("app.py syntax: valid")

# The checkpoint is trusted project output created by your own training run.
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint keys:", list(checkpoint.keys()))
print("Architecture:", checkpoint.get("architecture"))
print("Best epoch:", checkpoint.get("epoch"))
print("Best validation loss:", checkpoint.get("best_val_loss"))
print("Image size:", checkpoint.get("image_size"))
print("Normalization:", checkpoint.get("normalization"))

assert checkpoint.get("architecture") == "ResNet34UNet"
assert checkpoint.get("image_size") == 352
assert checkpoint.get("normalization") == "ImageNet"
print("Checkpoint metadata verified.")

app.py syntax: valid
Checkpoint keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'best_val_loss', 'architecture', 'image_size', 'normalization']
Architecture: ResNet34UNet
Best epoch: 8
Best validation loss: 0.2517758094271024
Image size: 352
Normalization: ImageNet
Checkpoint metadata verified.


# **Architecture-to-checkpoint compatibility**

In [49]:
import importlib
import models.unet

importlib.reload(models.unet)
from models.unet import ResNet34UNet

verification_model = ResNet34UNet(pretrained=False)
load_result = verification_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)
verification_model.eval()

print("Strict checkpoint loading successful:", load_result)
del verification_model

Strict checkpoint loading successful: <All keys matched successfully>


# **Smoke test**

In [50]:
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
smoke_model = ResNet34UNet(pretrained=False)
smoke_model.load_state_dict(checkpoint["model_state_dict"], strict=True)
smoke_model.to(device).eval()

dummy_input = torch.zeros(1, 3, 352, 352, dtype=torch.float32, device=device)

with torch.inference_mode():
    dummy_logits = smoke_model(dummy_input)
    dummy_probabilities = torch.sigmoid(dummy_logits)

print("Device:", device)
print("Input shape:", tuple(dummy_input.shape))
print("Output shape:", tuple(dummy_logits.shape))
print("Probability range:",
      float(dummy_probabilities.min()),
      "to",
      float(dummy_probabilities.max()))

assert tuple(dummy_logits.shape) == (1, 1, 352, 352)
print("Inference smoke test passed.")

del smoke_model, dummy_input, dummy_logits, dummy_probabilities
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Device: cuda
Input shape: (1, 3, 352, 352)
Output shape: (1, 1, 352, 352)
Probability range: 7.961166448300361e-11 to 7.06040097497862e-08
Inference smoke test passed.


# **Launch the Gradio app**

In [51]:
import importlib
import app

# If app.py was edited during this session, reload it before launching.
app = importlib.reload(app)

# Avoid launching duplicate servers in the same runtime.
if getattr(app, "demo", None) is None:
    raise RuntimeError("Gradio demo was not created. Check app.py.")

launch_result = app.demo.launch(
    share=True,
    debug=False,
    prevent_thread_lock=True,
    show_error=True
)

print("Gradio launch result:", launch_result)

Loading ResNet34 U-Net...
Model loaded successfully!
Device: cuda
Checkpoint epoch: 8
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/11/28 09:10:29 [W] [service.go:132] login to server failed: session shutdown


<IPython.core.display.Javascript object>

Gradio launch result: 
